In [ ]:
"""Complete training pipeline with dynamic YOLO11 model support."""

import os
import sys
from pathlib import Path
from typing import List, Dict, Optional  # Add this import for type hints

# Import the new dynamic trainer
from abbvisionsystem.training_pipeline.yolo_trainer import (
    YOLO11DefectDetector, 
    compare_yolo11_models,
    evaluate_yolo11_model
)
from abbvisionsystem.training_pipeline.data_manager import (
    organize_dataset, 
    prepare_yolo_dataset, 
    augment_with_backgrounds,
    prepare_yolo_dataset_from_realistic
)
from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel


def run_dynamic_pipeline(
    source_data_dir: str,
    model_variants: List[str] = ['yolo11s'],  # Default to recommended model
    train_epochs: int = 75,
    compare_models: bool = False,
    use_classification: bool = False
):
    """
    Run complete training pipeline with dynamic YOLO11 model support.
    
    Args:
        source_data_dir: Path to source data directory
        model_variants: List of YOLO11 variants to train/compare
        train_epochs: Number of training epochs
        compare_models: Whether to compare multiple models
        use_classification: Whether to also train classification model
    """
    
    print("🚀 Starting Dynamic YOLO11 Defect Detection Pipeline")
    print("=" * 70)
    
    # Display available models
    print(f"\n📋 Available YOLO11 Models:")
    available_models = YOLO11DefectDetector.list_available_models()
    for variant, info in available_models.items():
        if info['specs']:
            print(f"   {variant}: {info['specs']['size']} - {info['specs']['use_case']}")
    
    # Validate requested models
    valid_variants = []
    for variant in model_variants:
        if variant in available_models:
            valid_variants.append(variant)
            info = available_models[variant]
            print(f"✅ {variant} - {info['type']} model ({info['specs'].get('use_case', 'Unknown use case')})")
        else:
            print(f"❌ {variant} - Unknown model variant")
    
    if not valid_variants:
        print("❌ No valid model variants specified")
        return None
    
    model_variants = valid_variants
    
    # Step 1: Organize dataset
    print("\n📁 Step 1: Organizing dataset...")
    classification_dataset = "defect_detection_dataset"
    organize_dataset(source_data_dir, classification_dataset)
    
    # Step 2: Create realistic training data
    print("\n🎨 Step 2: Creating realistic training data...")
    realistic_train_dir = "realistic_training_data"
    augment_with_backgrounds(
        source_data_dir,
        realistic_train_dir,
        objects_per_image=(1, 4),
        images_per_object=5,
        multi_object_scenes=150
    )
    
    # Step 3: Prepare YOLO dataset
    print("\n🎯 Step 3: Preparing YOLO11 dataset...")
    yolo_dataset_yaml = prepare_yolo_dataset_from_realistic(
        realistic_train_dir, "yolo11_dataset"
    )
    
    results = {}
    
    # Step 4: Train models
    if compare_models and len(model_variants) > 1:
        print(f"\n🔬 Step 4: Comparing {len(model_variants)} YOLO11 models...")
        
        comparison_results = compare_yolo11_models(
            dataset_yaml=yolo_dataset_yaml,
            model_variants=model_variants,
            epochs=train_epochs,
            test_images_dir=f"{source_data_dir}/both" if os.path.exists(f"{source_data_dir}/both") else None
        )
        
        results['model_comparison'] = comparison_results
        
        # Find best performing model
        best_model = None
        best_detection_rate = 0
        
        for variant, result in comparison_results.items():
            if result.get('status') == 'failed':
                continue
            
            detection_rate = result.get('detection_rate', 0)
            if detection_rate > best_detection_rate:
                best_detection_rate = detection_rate
                best_model = variant
        
        if best_model:
            print(f"\n🏆 Best performing model: {best_model} ({best_detection_rate:.2%} detection rate)")
            results['best_model'] = best_model
        
    else:
        # Train single model or multiple models individually
        print(f"\n🤖 Step 4: Training YOLO11 models...")
        
        for variant in model_variants:
            print(f"\n🔧 Training {variant.upper()}...")
            
            try:
                # Create detector with specified variant
                detector = YOLO11DefectDetector(model_variant=variant)
                
                # Display model info
                info = detector.get_model_info()
                print(f"ℹ️  Model info: {info['type']} model, {info['specs'].get('size', 'Unknown size')}")
                
                # Train model
                best_weights = detector.train(
                    dataset_yaml=yolo_dataset_yaml,
                    epochs=train_epochs,
                    project='yolo11_trained_models',
                    name=f'{variant}_defect_detector'
                )
                
                # Evaluate on test data if available
                test_dir = f"{source_data_dir}/both"
                if os.path.exists(test_dir):
                    print(f"📊 Evaluating {variant} on test data...")
                    eval_results = evaluate_yolo11_model(detector, test_dir)
                    results[variant] = eval_results
                    
                    print(f"✅ {variant}: "
                          f"{eval_results.get('detection_rate', 0):.2%} detection rate, "
                          f"{eval_results.get('avg_inference_time', 0):.3f}s avg inference")
                else:
                    results[variant] = {
                        'status': 'trained',
                        'weights_path': best_weights,
                        'model_info': info
                    }
                
            except Exception as e:
                print(f"❌ {variant} training failed: {e}")
                results[variant] = {'status': 'failed', 'error': str(e)}
    
    # Step 5: Optional classification model for comparison
    if use_classification:
        print("\n🧠 Step 5: Training ResNet50V2 classification model for comparison...")
        
        try:
            classifier = DefectClassificationModel()
            classifier.build_model()
            
            train_gen, val_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/train",
                f"{classification_dataset}/validation"
            )
            
            classifier.train(
                train_gen, val_gen,
                epochs=max(30, train_epochs // 2),  # Fewer epochs for classification
                model_name="resnet_defect_classifier"
            )
            
            test_datagen = classifier.prepare_data_generators(
                f"{classification_dataset}/test",
                f"{classification_dataset}/test"
            )[1]
            
            classification_results = classifier.evaluate(test_datagen)
            results['classification'] = classification_results
            
            classifier.save_model("resnet_defect_classifier")
            
            print(f"✅ Classification Results:")
            print(f"   Accuracy: {classification_results['test_accuracy']:.4f}")
            print(f"   Precision: {classification_results['test_precision']:.4f}")
            print(f"   Recall: {classification_results['test_recall']:.4f}")
            
        except Exception as e:
            print(f"❌ Classification training failed: {e}")
            results['classification'] = {'status': 'failed', 'error': str(e)}
    
    # Step 6: Generate comprehensive report
    print("\n📈 Step 6: Generating Results Summary")
    print("=" * 50)
    
    generate_dynamic_pipeline_report(results, model_variants, compare_models)
    
    print("\n✅ Dynamic YOLO11 Pipeline completed successfully!")
    
    return results


def generate_dynamic_pipeline_report(results: Dict, model_variants: List[str], comparison_mode: bool):
    """Generate comprehensive report for dynamic pipeline results."""
    
    print(f"\n📊 YOLO11 DYNAMIC PIPELINE RESULTS")
    print("=" * 50)
    
    if comparison_mode and 'model_comparison' in results:
        print(f"\n🔬 MODEL COMPARISON RESULTS:")
        comparison = results['model_comparison']
        
        print(f"{'Model':<15} {'Status':<10} {'Detection Rate':<15} {'Avg Inference':<15}")
        print("-" * 65)
        
        for variant, result in comparison.items():
            status = result.get('status', 'completed')
            detection_rate = result.get('detection_rate', 0)
            inference_time = result.get('avg_inference_time', 0)
            
            print(f"{variant:<15} {status:<10} {detection_rate:<15.2%} {inference_time:<15.3f}s")
        
        if 'best_model' in results:
            print(f"\n🏆 Recommended model: {results['best_model']}")
    
    else:
        print(f"\n🤖 INDIVIDUAL MODEL RESULTS:")
        
        for variant in model_variants:
            if variant in results:
                result = results[variant]
                status = result.get('status', 'completed')
                
                print(f"\n{variant.upper()}:")
                print(f"   Status: {status}")
                
                if status != 'failed':
                    if 'detection_rate' in result:
                        print(f"   Detection Rate: {result['detection_rate']:.2%}")
                        print(f"   Avg Inference Time: {result.get('avg_inference_time', 0):.3f}s")
                        print(f"   Total Detections: {result.get('total_detections', 0)}")
                    
                    if 'weights_path' in result:
                        print(f"   Weights: {result['weights_path']}")
                else:
                    print(f"   Error: {result.get('error', 'Unknown error')}")
    
    # Classification comparison if available
    if 'classification' in results:
        print(f"\n🧠 CLASSIFICATION MODEL COMPARISON:")
        cls_result = results['classification']
        
        if cls_result.get('status') != 'failed':
            print(f"   ResNet50V2 Accuracy: {cls_result.get('test_accuracy', 0):.4f}")
            print(f"   ResNet50V2 Precision: {cls_result.get('test_precision', 0):.4f}")
            print(f"   ResNet50V2 Recall: {cls_result.get('test_recall', 0):.4f}")
        else:
            print(f"   Classification training failed: {cls_result.get('error', 'Unknown error')}")
    
    # Recommendations
    print(f"\n💡 RECOMMENDATIONS:")
    
    # Find fastest and most accurate models from results
    fastest_model = None
    most_accurate_model = None
    fastest_time = float('inf')
    highest_accuracy = 0
    
    for variant in model_variants:
        if variant in results and results[variant].get('status') != 'failed':
            result = results[variant]
            
            inference_time = result.get('avg_inference_time', float('inf'))
            if inference_time < fastest_time:
                fastest_time = inference_time
                fastest_model = variant
            
            detection_rate = result.get('detection_rate', 0)
            if detection_rate > highest_accuracy:
                highest_accuracy = detection_rate
                most_accurate_model = variant
    
    if fastest_model:
        print(f"   ⚡ Fastest: {fastest_model} ({fastest_time:.3f}s)")
    
    if most_accurate_model:
        print(f"   🎯 Most Accurate: {most_accurate_model} ({highest_accuracy:.2%})")
    
    print(f"\n📋 DEPLOYMENT SUGGESTIONS:")
    print(f"   • Use YOLO11N/S for real-time production lines")
    print(f"   • Use YOLO11M/L for quality control stations")
    print(f"   • Use YOLO11-seg for detailed defect analysis")
    print(f"   • Use YOLO11-obb for rotated package detection")


def test_dynamic_setup():
    """Test if dynamic YOLO11 setup is working properly."""
    print("🔍 Testing dynamic YOLO11 setup...")
    
    try:
        # Test basic import - Fix the import path
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLO11DefectDetector
        print("✅ YOLO11DefectDetector import successful")
        
        # Test model listing
        available_models = YOLO11DefectDetector.list_available_models()
        print(f"✅ Found {len(available_models)} available model variants")
        
        # Test model recommendations
        recommendations = YOLO11DefectDetector.recommend_model_for_use_case('production')
        print(f"✅ Production recommendations: {recommendations}")
        
        # Test model info
        detector = YOLO11DefectDetector('yolo11s')
        info = detector.get_model_info()
        print(f"✅ Model info retrieval: {info['variant']} ({info['type']})")
        
        return True
        
    except Exception as e:
        print(f"❌ Dynamic setup test failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
        return False


# Quick fix function to validate imports
def validate_imports():
    """Validate all required imports are working."""
    try:
        print("🔍 Validating imports...")
        
        # Test typing imports
        from typing import List, Dict, Optional
        print("✅ typing imports successful")
        
        # Test basic imports
        import os, sys
        from pathlib import Path
        print("✅ Basic imports successful")
        
        # Test ultralytics
        from ultralytics import YOLO
        print("✅ ultralytics import successful")
        
        # Test your custom modules
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLO11DefectDetector
        print("✅ Custom YOLO11DefectDetector import successful")
        
        from abbvisionsystem.training_pipeline.data_manager import organize_dataset
        print("✅ data_manager imports successful")
        
        try:
            from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel
            print("✅ resnet_trainer import successful")
        except ImportError as e:
            print(f"⚠️  resnet_trainer import failed (optional): {e}")
        
        print("✅ All critical imports validated successfully!")
        return True
        
    except Exception as e:
        print(f"❌ Import validation failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
        return False


if __name__ == "__main__":
    # First validate imports
    print("=" * 70)
    print("STEP 0: Import Validation")
    print("=" * 70)
    
    if not validate_imports():
        print("❌ Import validation failed. Please fix the imports above.")
        exit(1)
    
    # Test dynamic setup
    print("\n" + "=" * 70)
    print("STEP 1: Dynamic Setup Test")
    print("=" * 70)
    
    if not test_dynamic_setup():
        print("❌ Dynamic setup test failed. Please fix the issues above.")
        exit(1)
    
    # Configuration
    source_dir = "data/choco-pie"  # Update this path
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"⚠️  Source directory not found: {source_dir}")
        print("Please update the source_dir variable to point to your data directory.")
        print("Available options:")
        for potential_dir in ["data", "dataset", "choco-pie", "images"]:
            if os.path.exists(potential_dir):
                print(f"   ✅ Found: {potential_dir}")
        
        # Use current directory if source not found
        print(f"\n📁 Current directory contents:")
        for item in os.listdir("."):
            if os.path.isdir(item):
                print(f"   📂 {item}/")
        
        # Ask user to continue with a found directory or exit
        print("\n⚠️  Update source_dir in the script and run again.")
        exit(1)
    
    # Example 1: Train single recommended model
    print("\n" + "="*70)
    print("EXAMPLE 1: Single Model Training (Recommended)")
    print("="*70)
    
    try:
        results_single = run_dynamic_pipeline(
            source_data_dir=source_dir,
            model_variants=['yolo11s'],  # Recommended model
            train_epochs=50,
            compare_models=False,
            use_classification=False
        )
        print("✅ Example 1 completed successfully!")
    except Exception as e:
        print(f"❌ Example 1 failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
    
    # Example 2: Compare multiple models
    print("\n" + "="*70)
    print("EXAMPLE 2: Multi-Model Comparison (Research)")
    print("="*70)
    
    try:
        results_comparison = run_dynamic_pipeline(
            source_data_dir=source_dir,
            model_variants=['yolo11n', 'yolo11s', 'yolo11m'],  # Compare multiple
            train_epochs=30,  # Shorter for comparison
            compare_models=True,
            use_classification=True
        )
        print("✅ Example 2 completed successfully!")
    except Exception as e:
        print(f"❌ Example 2 failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
    
    # Example 3: Segmentation model for precise boundaries
    print("\n" + "="*70)
    print("EXAMPLE 3: Segmentation Model (Precise Boundaries)")
    print("="*70)
    
    try:
        results_segmentation = run_dynamic_pipeline(
            source_data_dir=source_dir,
            model_variants=['yolo11s-seg'],  # Segmentation model
            train_epochs=40,
            compare_models=False,
            use_classification=False
        )
        print("✅ Example 3 completed successfully!")
    except Exception as e:
        print(f"❌ Example 3 failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
    
    print("\n🎉 All dynamic pipeline examples completed!")
    print("Choose the configuration that best fits your needs:")
    print("  • Single model: Fast training, good for production")
    print("  • Comparison: Research-oriented, best model selection")
    print("  • Segmentation: Precise defect boundaries")

STEP 0: Import Validation
🔍 Validating imports...
✅ typing imports successful
✅ Basic imports successful
✅ ultralytics import successful
✅ Custom YOLO11DefectDetector import successful
✅ data_manager imports successful
✅ resnet_trainer import successful
✅ All critical imports validated successfully!

STEP 1: Dynamic Setup Test
🔍 Testing dynamic YOLO11 setup...
✅ YOLO11DefectDetector import successful
✅ Found 20 available model variants
✅ Production recommendations: ['yolo11s', 'yolo11m']
✅ Model info retrieval: yolo11s (detection)

EXAMPLE 1: Single Model Training (Recommended)
🚀 Starting Dynamic YOLO11 Defect Detection Pipeline

📋 Available YOLO11 Models:
   yolo11n: ~6MB - Real-time edge
   yolo11s: ~22MB - Production (Recommended)
   yolo11m: ~50MB - Quality control
   yolo11l: ~87MB - Research/offline
   yolo11x: ~137MB - Maximum accuracy
   yolo11n-seg: ~6MB - Real-time edge
   yolo11s-seg: ~22MB - Production (Recommended)
   yolo11m-seg: ~50MB - Quality control
   yolo11l-seg: ~8

train: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo11_dataset/labels/train... 323 images, 0 backgrounds, 0 corrupt: 100%|██████████| 323/323 [00:00<00:00, 3379.26it/s]

train: New cache created: /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo11_dataset/labels/train.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 780.6±245.7 MB/s, size: 174.4 KB)


val: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo11_dataset/labels/val... 151 images, 0 backgrounds, 0 corrupt: 100%|██████████| 151/151 [00:00<00:00, 3961.57it/s]

...


      25/50         0G     0.1526     0.2911     0.8821          9        640: 100%|██████████| 41/41 [02:07<00:00,  3.12s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [01:00<00:00,  6.00s/it]

                   all        151        159      0.999          1      0.995      0.993

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size



      26/50         0G     0.1445     0.3121     0.8888         13        640: 100%|██████████| 41/41 [02:02<00:00,  2.99s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [01:00<00:00,  6.03s/it]

                   all        151        159      0.999          1      0.995      0.995

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size



      27/50         0G     0.1486     0.2571     0.8881         19        640:  71%|███████   | 29/41 [01:32<00:38,  3.22s/it]